## Learning Objectives

By the end of this tutorial, you will be able to

1. Load a preprocessed version of the UN General Debate Corpus in R.
2. Build and trim a document-feature matrix with `quanteda`.
3. Inspect a 40-topic Structured Topic Model (STM) fitted with `stm`.
4. Visualize topic prevalence and topic dynamics over time.

## Target audience

This tutorial is aimed at researchers and students with basic R knowledge who want to learn how to run and interpret a simple Structured Topic Model workflow for social science text data.

## Setting up the computational environment

This notebook uses the R kernel (`IRkernel`) together with the R packages `quanteda` and `stm`. In line with the Methods Hub guidelines, package installation is handled outside the notebook. The repository includes:

- `renv.lock` for reproducing the local R environment
- `binder/install.R` and `binder/runtime.txt` for Binder-based execution
- `binder/postBuild` for Methods Hub rendering with Quarto

The code below assumes that these dependencies are already installed.

## Duration

Around 25 minutes.

## Social Science Use Case(s)

Topic models are widely used in the social sciences to analyze large text collections such as parliamentary debates, party manifestos, news coverage, policy documents, and international speeches. Here we use the UN General Debate Corpus to explore how substantive topics are distributed across speeches by country and over time. This tutorial closely follows the original `un-stm.ipynb` notebook from the `stmdemo` repository and keeps the same dataset and model parametrization (`K = 40`).


## Load packages

We begin by loading the two packages used throughout the tutorial.


In [ ]:
library(quanteda)
library(stm)

## Load the data

We use the same dataset as in the original notebook: the UN General Debate Corpus with additional metadata. The object `uncorpus` contains the speeches, and `uncorpus.stats` contains document-level summary information.


In [ ]:
load("data/UNgeneraldebate.corpus.RData")
head(uncorpus.stats, 10)

## Create and trim the document-feature matrix

Next we calculate a document-feature matrix (DFM), a table in which rows are documents and columns are words. As in the source notebook, we remove numbers, punctuation, symbols, and standard English stop words. We then trim the DFM by excluding terms that are too rare or too common. Specifically, we keep terms that appear in at least 7.5% and at most 90% of documents.

Because current `quanteda` versions no longer accept the older `remove =` argument inside `dfm()`, we first tokenize the corpus and then remove stop words with `tokens_remove()`. This updates the original workflow while preserving the same preprocessing decisions.


In [ ]:
uncorpus.tokens <- tokens(
  uncorpus,
  remove_numbers = TRUE,
  remove_punct = TRUE,
  remove_symbols = TRUE
)
uncorpus.tokens <- tokens_remove(uncorpus.tokens, pattern = stopwords("english"))
uncorpus.dfm <- dfm(uncorpus.tokens)
uncorpus.dfm.trim <- dfm_trim(
  uncorpus.dfm,
  min_docfreq = 0.075,
  max_docfreq = 0.90,
  docfreq_type = "prop"
)
uncorpus.dfm.trim

## Fit the STM model

We now prepare the DFM for `stm` and inspect a model with `K = 40` topics, exactly as in the original notebook. To keep execution time manageable and results identical to the source material, we load a pre-fitted STM object from disk. The commented line shows the original fitting command with spectral initialization.


In [ ]:
topic.count <- 40
dfm2stm <- convert(uncorpus.dfm.trim, to = "stm")
# model.stm <- stm(dfm2stm$documents, dfm2stm$vocab, K = topic.count, data = dfm2stm$meta, init.type = "Spectral")
load("data/UNgeneraldebate.stm.RData")
as.data.frame(labelTopics(model.stm, n = 10)$prob)

## Visualize topic summaries

The `stm` package provides several built-in diagnostic and interpretive plots. We start with the overall topic summary plot, which shows the estimated prevalence of topics in the corpus.


In [ ]:
plot(model.stm, type = "summary", text.cex = 0.5)

A perspectives plot contrasts the vocabularies that distinguish two topics. As in the original notebook, we compare topics 16 and 21.


In [ ]:
plot(model.stm, type = "perspectives", topics = c(16, 21))

A histogram plot shows the distribution of topic proportions across documents. We set a random seed so that the sampled topics are reproducible.


In [ ]:
set.seed(123)
plot(model.stm, type = "hist", topics = sample(1:topic.count, size = 9))

## Estimate topic prevalence over time

To examine how topic prevalence changes over time, we estimate topic effects using metadata. This follows the original notebook: we model topic prevalence as a function of country and a smooth term for year.


In [ ]:
model.stm.labels <- labelTopics(model.stm, 1:topic.count)
dfm2stm$meta$datum <- as.numeric(dfm2stm$meta$year)
model.stm.ee <- estimateEffect(1:topic.count ~ country + s(year), model.stm, meta = dfm2stm$meta)

The next plot displays estimated topic prevalence over time for nine sampled topics. Again, we fix the seed for reproducibility.


In [ ]:
set.seed(123)
selected_topics <- sample(1:topic.count, size = 9)
par(mfrow = c(3, 3))
for (topic_id in selected_topics) {
  plot(
    model.stm.ee,
    "year",
    method = "continuous",
    topics = topic_id,
    main = paste0(model.stm.labels$prob[topic_id, 1:3], collapse = ", "),
    printlegend = FALSE
  )
}

## Save topic-prevalence plots for all 40 topics

If you want to export the prevalence plots for all topics, you can use the code below. It is left commented out, following the source notebook.


In [ ]:
# png("all-topic-prevalence.png", width = 800, height = 800)
# for (i in 1:topic.count) {
#   plot(
#     model.stm.ee,
#     "year",
#     method = "continuous",
#     topics = i,
#     main = paste0(model.stm.labels$prob[i, 1:3], collapse = ", "),
#     printlegend = FALSE
#   )
# }
# dev.off()

## Conclusion

In this tutorial, you reproduced a complete STM workflow for UN General Debate speeches in R. You loaded the corpus, created and trimmed a DFM, inspected a 40-topic STM, and visualized both topic structure and topic prevalence over time. The tutorial preserves the original dataset and core parametrization of the source notebook while updating the preprocessing code to work with current `quanteda` versions.

## References

- Puschmann, C. *Topic modeling with quanteda and STM*. Original notebook: http://rpubs.com/cbpuschmann/un-stm and repository source: https://github.com/arnim/stmdemo/blob/master/un-stm.ipynb
- Roberts, M. E., Stewart, B. M., & Tingley, D. *stm: An R Package for Structural Topic Models*. Package website: https://www.structuraltopicmodel.com/
- Benoit, K., Watanabe, K., Wang, H., Nulty, P., Obeng, A., Müller, S., & Matsuo, A. *quanteda: An R package for the quantitative analysis of textual data*. Package website: https://quanteda.io/
- UN General Debate Corpus project page: http://www.smikhaylov.net/ungdc/
